In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
# Load dataset
df = pd.read_csv('tensile_strength_data.csv')

# Preview dataset
print(df.head())
#statistical info
print(df.info())
print(df.describe())

In [ ]:
sns.boxplot(x='Temperature', y='Tensile_Strength', hue='Pressure', data=df)
plt.title('Tensile Strength by Temperature and Pressure')
plt.ylabel('Tensile Strength (MPa)')
plt.show()

In [ ]:
# One-Way ANOVA (Effect of Temperature)
model_1way = ols('Tensile_Strength ~ C(Temperature)', data=df).fit()
anova_1way = sm.stats.anova_lm(model_1way, typ=2)
print(anova_1way)

# Interpretation
# If p-value < 0.05, temperature significantly affects tensile strength.
# F-statistic = ratio of between-group to within-group variation.

In [ ]:
#If ANOVA is significant, use Tukey’s test to find which temperature pairs differ.

tukey_result = pairwise_tukeyhsd(df['Tensile_Strength'], df['Temperature'], alpha=0.05)
print(tukey_result)

# The Tukey summary shows which pairs of temperatures differ significantly (based on mean strength).

In [ ]:
# Two-Way ANOVA (Temperature × Pressure)
model_2way = ols('Tensile_Strength ~ C(Temperature) * C(Pressure)', data=df).fit()
anova_2way = sm.stats.anova_lm(model_2way, typ=2)
print(anova_2way)

# Interpretation
# C(Temperature): main effect of temperature
# C(Pressure): main effect of pressure
# C(Temperature):C(Pressure): interaction between the two factors


In [ ]:
# Interaction Plot
# If lines are not parallel → interaction effect exists.

sns.pointplot(x='Temperature', y='Tensile_Strength', hue='Pressure', data=df,
             errorbar='sd', dodge=True, markers=['o', 's'], capsize=.1)
plt.title('Interaction between Temperature and Pressure')
plt.ylabel('Tensile Strength (MPa)')
plt.show()

In [ ]:
# Normality Test (Shapiro-Wilk)
# if p > 0.05 → residuals are normally distributed.

residuals = model_2way.resid
shapiro_test = stats.shapiro(residuals)
print("Shapiro-Wilk Test:", shapiro_test)


In [ ]:
# Homogeneity of Variance (Levene’s Test)
# p > 0.05 → equal variances assumption holds.

levene_test = stats.levene(
    df[df['Pressure'] == 'Low']['Tensile_Strength'],
    df[df['Pressure'] == 'High']['Tensile_Strength']
)
print("Levene’s Test:", levene_test)
